# DoS and GEAR Error Analysis with and without `id`

This notebook isolates the hypothesis that `id` is necessary to separate ambiguous CAN frames, especially for DoS and GEAR.

Experiments included:
- Dataset inspection for all-zero payloads
- MLP with `bytes only`
- MLP with `id + bytes`
- False-positive analysis for `DoS` and `GEAR`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

DATA_PATH = Path('../data/processed/carhacking_ciciov_benign_dos_aligned.csv')
BENIGN_TARGET = 1_000_000
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
df = df[['id', 'val0', 'val1', 'val2', 'val3', 'val4', 'val5', 'val6', 'val7', 'flag']].dropna().copy()
byte_cols = [f'val{i}' for i in range(8)]
df.head()

,id,val0,val1,val2,val3,val4,val5,val6,val7,flag
0,790,5,33,104,9,33,33,0,111,BENIGN
1,399,254,91,0,0,0,60,0,0,BENIGN
2,608,25,33,34,48,8,142,109,58,BENIGN
3,672,100,0,154,29,151,2,189,0,BENIGN
4,809,64,187,127,20,17,32,0,20,BENIGN


In [2]:
all_zero_mask = (df[byte_cols] == 0).all(axis=1)
zero_df = df.loc[all_zero_mask, ['id', 'flag']].copy()

print('all_zero_rows:', int(all_zero_mask.sum()))
print('all_zero_rate:', round(float(all_zero_mask.mean()), 4))
print()
print('all_zero_by_class')
print(zero_df['flag'].value_counts().to_string())
print()
print('all_zero_by_id0_and_class')
print(pd.crosstab(zero_df['id'] == 0, zero_df['flag']))

all_zero_rows: 2189909
all_zero_rate: 0.1161

all_zero_by_class
flag
BENIGN    1581282
DoS        587525
Fuzzy       21102

all_zero_by_id0_and_class
flag   BENIGN     DoS  Fuzzy
id                          
False  592410       4  21102
True   988872  587521      0


In [3]:
benign_df = df[df['flag'] == 'BENIGN']
attack_df = df[df['flag'] != 'BENIGN']
benign_sampled = benign_df.sample(n=min(BENIGN_TARGET, len(benign_df)), random_state=RANDOM_STATE)
df_balanced = pd.concat([benign_sampled, attack_df], axis=0)
df_balanced = df_balanced.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

print(df_balanced['flag'].value_counts())

flag
BENIGN    1000000
RPM        696373
DoS        693372
GEAR       637417
Fuzzy      579683
Name: count, dtype: int64


In [4]:
le = LabelEncoder()
df_balanced['flag_encoded'] = le.fit_transform(df_balanced['flag'])

X_with_id = df_balanced[['id'] + byte_cols].copy()
X_bytes_only = df_balanced[byte_cols].copy()
y = df_balanced['flag_encoded'].copy()

X_train_with_id, X_test_with_id, y_train, y_test = train_test_split(
    X_with_id, y, test_size=0.2, stratify=y, random_state=40
)

X_train_bytes = X_bytes_only.loc[X_train_with_id.index]
X_test_bytes = X_bytes_only.loc[X_test_with_id.index]

print('with_id train/test:', X_train_with_id.shape, X_test_with_id.shape)
print('bytes_only train/test:', X_train_bytes.shape, X_test_bytes.shape)

with_id train/test: (2885476, 9) (721369, 9)
bytes_only train/test: (2885476, 8) (721369, 8)


In [5]:
def train_mlp(X_train, X_test, y_train, y_test, label):
    model = MLPClassifier(
        hidden_layer_sizes=(30, 30, 30),
        activation='relu',
        solver='adam',
        learning_rate='adaptive',
        learning_rate_init=0.001,
        max_iter=25,
        random_state=90,
        early_stopping=True
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    print(f'===== {label} =====')
    print(classification_report(y_test, pred, target_names=le.classes_.astype(str), digits=5))
    return model, pred, proba

mlp_bytes_model, mlp_bytes_pred, mlp_bytes_proba = train_mlp(
    X_train_bytes, X_test_bytes, y_train, y_test, 'MLP - bytes only'
)

mlp_id_model, mlp_id_pred, mlp_id_proba = train_mlp(
    X_train_with_id, X_test_with_id, y_train, y_test, 'MLP - id + bytes'
)

===== MLP - bytes only =====
              precision    recall  f1-score   support

      BENIGN    0.99959   0.90346   0.94909    200000
         DoS    0.84955   0.95499   0.89919    138674
       Fuzzy    0.99980   0.94050   0.96924    115937
        GEAR    1.00000   0.93642   0.96717    127483
         RPM    0.89118   1.00000   0.94246    139275

    accuracy                        0.94378    721369
   macro avg    0.94803   0.94707   0.94543    721369
weighted avg    0.94992   0.94378   0.94465    721369

===== MLP - id + bytes =====
              precision    recall  f1-score   support

      BENIGN    0.99957   0.93784   0.96772    200000
         DoS    0.91551   0.95499   0.93483    138674
       Fuzzy    0.99896   0.94045   0.96882    115937
        GEAR    1.00000   0.93642   0.96717    127483
         RPM    0.86754   1.00000   0.92907    139275

    accuracy                        0.95331    721369
   macro avg    0.95631   0.95394   0.95352    721369
weighted avg    0.9

In [6]:
def build_error_frame(X_test_reference, y_true, y_pred):
    out = X_test_reference.copy()
    out['true_label'] = le.inverse_transform(y_true)
    out['pred_label'] = le.inverse_transform(y_pred)
    return out

analysis_bytes = build_error_frame(X_test_with_id, y_test.to_numpy(), mlp_bytes_pred)
analysis_with_id = build_error_frame(X_test_with_id, y_test.to_numpy(), mlp_id_pred)

In [7]:
def summarize_false_positives(analysis_df, target_label, title):
    fp = analysis_df[(analysis_df['pred_label'] == target_label) & (analysis_df['true_label'] != target_label)].copy()
    tp = analysis_df[(analysis_df['pred_label'] == target_label) & (analysis_df['true_label'] == target_label)].copy()

    print(f'===== {title} | {target_label} =====')
    print('false_positives:', len(fp))
    print('true_positives:', len(tp))

    if fp.empty:
        print('Sem falsos positivos para essa classe.')
        print()
        return

    print('actual_class_breakdown')
    print(fp['true_label'].value_counts().to_string())
    print('actual_class_breakdown_pct')
    print((fp['true_label'].value_counts(normalize=True) * 100).round(2).to_string())

    fp_all_zero = (fp[byte_cols] == 0).all(axis=1)
    print('fp_all_zero_rate:', round(float(fp_all_zero.mean()), 4))
    print('fp_id_zero_rate:', round(float((fp['id'] == 0).mean()), 4))

    if len(tp):
        tp_all_zero = (tp[byte_cols] == 0).all(axis=1)
        print('tp_all_zero_rate:', round(float(tp_all_zero.mean()), 4))
        print('tp_id_zero_rate:', round(float((tp['id'] == 0).mean()), 4))

    print('top_fp_ids')
    print(fp['id'].value_counts().head(10).to_string())

    print('top_fp_signatures')
    top_patterns = fp[['id'] + byte_cols + ['true_label']].value_counts().head(10)
    print(top_patterns.to_string())
    print()

summarize_false_positives(analysis_bytes, 'DoS', 'bytes only')
summarize_false_positives(analysis_bytes, 'GEAR', 'bytes only')
summarize_false_positives(analysis_with_id, 'DoS', 'id + bytes')
summarize_false_positives(analysis_with_id, 'GEAR', 'id + bytes')

===== bytes only | DoS =====
false_positives: 23452
true_positives: 132432
actual_class_breakdown
true_label
BENIGN    19287
Fuzzy      4165
actual_class_breakdown_pct
true_label
BENIGN    82.24
Fuzzy     17.76
fp_all_zero_rate: 1.0
fp_id_zero_rate: 0.5211
tp_all_zero_rate: 0.887
tp_id_zero_rate: 0.8869
top_fp_ids
id
0       12222
1072     4730
1520     4165
1201     1950
610       114
1109       98
1440       96
951        18
986        13
902        12
top_fp_signatures
id    val0  val1  val2  val3  val4  val5  val6  val7  true_label
0     0     0     0     0     0     0     0     0     BENIGN        12222
1072  0     0     0     0     0     0     0     0     BENIGN         4730
1520  0     0     0     0     0     0     0     0     Fuzzy          4165
1201  0     0     0     0     0     0     0     0     BENIGN         1950
610   0     0     0     0     0     0     0     0     BENIGN          114
1109  0     0     0     0     0     0     0     0     BENIGN           98
1440  0     0 

In [8]:
for title, analysis_df, pred in [
    ('bytes only', analysis_bytes, mlp_bytes_pred),
    ('id + bytes', analysis_with_id, mlp_id_pred),
]:
    print(f'===== confusion summary: {title} =====')
    cm = confusion_matrix(analysis_df['true_label'], analysis_df['pred_label'], labels=le.classes_)
    cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
    print(cm_df.loc[['BENIGN', 'DoS', 'GEAR'], ['BENIGN', 'DoS', 'GEAR']])
    print()

===== confusion summary: bytes only =====
        BENIGN     DoS    GEAR
BENIGN  180691   19287       0
DoS          0  132432       0
GEAR         0       0  119378

===== confusion summary: id + bytes =====
        BENIGN     DoS    GEAR
BENIGN  187568   12222       0
DoS          0  132432       0
GEAR         0       0  119378



## Context Features Experiment
Test if simple context features built from train statistics reduce DoS false positives.
Features added:
- `is_all_zero`
- `id_is_zero`
- `sum_bytes`
- `nonzero_bytes`
- `id_count_log` (frequency of ID in train)
- `id_zero_rate` (rate of all-zero payload for ID in train)

In [9]:
train_all_zero = (X_train_with_id[byte_cols] == 0).all(axis=1)
id_count_map = X_train_with_id['id'].value_counts()
id_zero_rate_map = train_all_zero.groupby(X_train_with_id['id']).mean()

def add_context_features(X):
    out = X.copy()
    out['is_all_zero'] = (out[byte_cols] == 0).all(axis=1).astype(int)
    out['id_is_zero'] = (out['id'] == 0).astype(int)
    out['sum_bytes'] = out[byte_cols].sum(axis=1)
    out['nonzero_bytes'] = (out[byte_cols] != 0).sum(axis=1)
    out['id_count_log'] = np.log1p(out['id'].map(id_count_map).fillna(0))
    out['id_zero_rate'] = out['id'].map(id_zero_rate_map).fillna(0)
    return out

X_train_context = add_context_features(X_train_with_id)
X_test_context = add_context_features(X_test_with_id)

print('context train/test:', X_train_context.shape, X_test_context.shape)
X_train_context.head()

context train/test: (2885476, 15) (721369, 15)


,id,val0,val1,val2,val3,val4,val5,val6,val7,is_all_zero,id_is_zero,sum_bytes,nonzero_bytes,id_count_log,id_zero_rate
1164644,1520,1,0,0,0,0,0,0,0,0,0,1,1,11.678321,0.143552
1832156,399,254,93,0,0,0,60,0,0,0,0,407,3,10.560852,0.000000
428885,497,8,0,0,0,0,0,0,0,0,0,8,1,9.866305,0.000000
3185857,1087,1,69,96,255,107,0,0,0,0,0,528,5,13.154011,0.000000
435490,790,69,41,36,255,41,36,0,255,0,0,733,7,13.239501,0.000000


In [10]:
mlp_context_model, mlp_context_pred, mlp_context_proba = train_mlp(
    X_train_context, X_test_context, y_train, y_test, 'MLP - id + bytes + context'
)

analysis_context = build_error_frame(X_test_with_id, y_test.to_numpy(), mlp_context_pred)

summarize_false_positives(analysis_context, 'DoS', 'id + bytes + context')
summarize_false_positives(analysis_context, 'GEAR', 'id + bytes + context')

print('===== confusion summary: id + bytes + context =====')
cm_context = confusion_matrix(analysis_context['true_label'], analysis_context['pred_label'], labels=le.classes_)
cm_context_df = pd.DataFrame(cm_context, index=le.classes_, columns=le.classes_)
print(cm_context_df.loc[['BENIGN', 'DoS', 'GEAR'], ['BENIGN', 'DoS', 'GEAR']])

===== MLP - id + bytes + context =====
              precision    recall  f1-score   support

      BENIGN    0.99971   0.93851   0.96815    200000
         DoS    0.91551   0.95497   0.93482    138674
       Fuzzy    0.99934   0.97663   0.98786    115937
        GEAR    1.00000   0.93642   0.96717    127483
         RPM    0.89120   1.00000   0.94247    139275

    accuracy                        0.95931    721369
   macro avg    0.96115   0.96131   0.96009    721369
weighted avg    0.96256   0.95931   0.95978    721369

===== id + bytes + context | DoS =====
false_positives: 12222
true_positives: 132430
actual_class_breakdown
true_label
BENIGN    12222
actual_class_breakdown_pct
true_label
BENIGN    100.0
fp_all_zero_rate: 1.0
fp_id_zero_rate: 1.0
tp_all_zero_rate: 0.887
tp_id_zero_rate: 0.887
top_fp_ids
id
0    12222
top_fp_signatures
id  val0  val1  val2  val3  val4  val5  val6  val7  true_label
0   0     0     0     0     0     0     0     0     BENIGN        12222

===== id + byt

## Remove Ambiguous BENIGN Rows Experiment
Remove rows where:
- `flag == 'BENIGN'`
- `id == 0`
- all bytes are zero

Then retrain and compare DoS/GEAR false positives.

In [14]:
ambiguous_benign_mask = (
    (df['flag'] == 'BENIGN') &
    (df['id'] == 0) &
    (df[byte_cols] == 0).all(axis=1)
)

removed_count = int(ambiguous_benign_mask.sum())
df_filtered = df.loc[~ambiguous_benign_mask].copy()

print('removed_ambiguous_benign_rows:', removed_count)
print('original_shape:', df.shape)
print('filtered_shape:', df_filtered.shape)
print()
print('filtered class counts')
print(df_filtered['flag'].value_counts())

removed_ambiguous_benign_rows: 988872
original_shape: (18856747, 10)
filtered_shape: (17867875, 10)

filtered class counts
flag
BENIGN    15261030
RPM         696373
DoS         693372
GEAR        637417
Fuzzy       579683
Name: count, dtype: int64


In [15]:
benign_df_f = df_filtered[df_filtered['flag'] == 'BENIGN']
attack_df_f = df_filtered[df_filtered['flag'] != 'BENIGN']
benign_sampled_f = benign_df_f.sample(n=min(BENIGN_TARGET, len(benign_df_f)), random_state=RANDOM_STATE)
df_balanced_f = pd.concat([benign_sampled_f, attack_df_f], axis=0)
df_balanced_f = df_balanced_f.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

df_balanced_f['flag_encoded'] = le.transform(df_balanced_f['flag'])

X_f = df_balanced_f[['id'] + byte_cols].copy()
y_f = df_balanced_f['flag_encoded'].copy()

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_f, y_f, test_size=0.2, stratify=y_f, random_state=40
)

mlp_filtered_model, mlp_filtered_pred, mlp_filtered_proba = train_mlp(
    X_train_f, X_test_f, y_train_f, y_test_f, 'MLP - id + bytes (filtered BENIGN ambiguous)'
)

analysis_filtered = build_error_frame(X_test_f, y_test_f.to_numpy(), mlp_filtered_pred)

summarize_false_positives(analysis_filtered, 'DoS', 'id + bytes + filtered BENIGN')
summarize_false_positives(analysis_filtered, 'GEAR', 'id + bytes + filtered BENIGN')

print('===== confusion summary: id + bytes + filtered BENIGN =====')
cm_f = confusion_matrix(analysis_filtered['true_label'], analysis_filtered['pred_label'], labels=le.classes_)
cm_f_df = pd.DataFrame(cm_f, index=le.classes_, columns=le.classes_)
print(cm_f_df.loc[['BENIGN', 'DoS', 'GEAR'], ['BENIGN', 'DoS', 'GEAR']])

===== MLP - id + bytes (filtered BENIGN ambiguous) =====
              precision    recall  f1-score   support

      BENIGN    0.99955   0.99918   0.99937    200000
         DoS    1.00000   0.95499   0.97698    138674
       Fuzzy    0.99953   0.94012   0.96892    115937
        GEAR    0.99997   0.93642   0.96716    127483
         RPM    0.86731   1.00000   0.92894    139275

    accuracy                        0.97026    721369
   macro avg    0.97327   0.96614   0.96827    721369
weighted avg    0.97418   0.97026   0.97088    721369

===== id + bytes + filtered BENIGN | DoS =====
false_positives: 0
true_positives: 132432
Sem falsos positivos para essa classe.

===== id + bytes + filtered BENIGN | GEAR =====
false_positives: 3
true_positives: 119378
actual_class_breakdown
true_label
BENIGN    3
actual_class_breakdown_pct
true_label
BENIGN    100.0
fp_all_zero_rate: 0.0
fp_id_zero_rate: 0.0
tp_all_zero_rate: 0.0
tp_id_zero_rate: 0.0
top_fp_ids
id
1087    2
1126    1
top_fp_signatur

## Export Filtered Dataset
Save the filtered dataset (without ambiguous BENIGN rows) to `data/processed` with a versioned file name and metadata summary.

In [13]:
from pathlib import Path
import json
from datetime import datetime

processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filtered_csv_path = processed_dir / f'carhacking_ciciov_benign_dos_filtered_no_ambiguous_benign_{stamp}.csv'
meta_json_path = processed_dir / f'carhacking_ciciov_benign_dos_filtered_no_ambiguous_benign_{stamp}_meta.json'

df_filtered.to_csv(filtered_csv_path, index=False)

meta = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'source_file': str(DATA_PATH),
    'output_file': str(filtered_csv_path),
    'rows_original': int(len(df)),
    'rows_filtered': int(len(df_filtered)),
    'rows_removed': int(removed_count),
    'removed_rule': "flag == 'BENIGN' and id == 0 and val0..val7 all zero",
    'class_counts_filtered': {k: int(v) for k, v in df_filtered['flag'].value_counts().to_dict().items()},
}

with open(meta_json_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

print('Saved CSV:', filtered_csv_path)
print('Saved meta:', meta_json_path)
print('Filtered class counts:')
print(df_filtered['flag'].value_counts().to_string())

Saved CSV: ..\data\processed\carhacking_ciciov_benign_dos_filtered_no_ambiguous_benign_20260707_111105.csv
Saved meta: ..\data\processed\carhacking_ciciov_benign_dos_filtered_no_ambiguous_benign_20260707_111105_meta.json
Filtered class counts:
flag
BENIGN    15261030
RPM         696373
DoS         693372
GEAR        637417
Fuzzy       579683


## CAN ID as Categorical (One-Hot Encoding)
Treat `id` as a categorical variable with `OneHotEncoder`, keep byte columns as numeric, and train a linear classifier on sparse features.

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import SGDClassifier

onehot_preprocessor = ColumnTransformer(
    transformers=[
        ('id_ohe', OneHotEncoder(handle_unknown='ignore'), ['id']),
        ('bytes', 'passthrough', byte_cols),
    ]
)

onehot_model = SGDClassifier(
    loss='log_loss',
    alpha=1e-5,
    max_iter=30,
    tol=1e-3,
    random_state=42,
    n_jobs=-1,
)

onehot_pipeline = Pipeline([
    ('prep', onehot_preprocessor),
    ('clf', onehot_model),
])

# Usa o mesmo split do experimento id+bytes para comparacao justa
onehot_pipeline.fit(X_train_with_id[['id'] + byte_cols], y_train)
onehot_pred = onehot_pipeline.predict(X_test_with_id[['id'] + byte_cols])

print('===== OneHot(id) + bytes | SGDClassifier =====')
print(classification_report(y_test, onehot_pred, target_names=le.classes_.astype(str), digits=5))

analysis_onehot = build_error_frame(X_test_with_id, y_test.to_numpy(), onehot_pred)
summarize_false_positives(analysis_onehot, 'DoS', 'onehot(id) + bytes')
summarize_false_positives(analysis_onehot, 'GEAR', 'onehot(id) + bytes')

print('===== confusion summary: onehot(id) + bytes =====')
cm_oh = confusion_matrix(analysis_onehot['true_label'], analysis_onehot['pred_label'], labels=le.classes_)
cm_oh_df = pd.DataFrame(cm_oh, index=le.classes_, columns=le.classes_)
print(cm_oh_df.loc[['BENIGN', 'DoS', 'GEAR'], ['BENIGN', 'DoS', 'GEAR']])

c:\Users\thelo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


===== OneHot(id) + bytes | SGDClassifier =====
              precision    recall  f1-score   support

      BENIGN    0.96714   0.91519   0.94045    200000
         DoS    0.88822   0.95499   0.92040    138674
       Fuzzy    0.95875   0.88510   0.92045    115937
        GEAR    0.99727   0.93642   0.96589    127483
         RPM    0.89119   1.00000   0.94247    139275

    accuracy                        0.93813    721369
   macro avg    0.94051   0.93834   0.93793    721369
weighted avg    0.94128   0.93813   0.93826    721369

===== onehot(id) + bytes | DoS =====
false_positives: 16666
true_positives: 132432
actual_class_breakdown
true_label
BENIGN    12222
Fuzzy      4444
actual_class_breakdown_pct
true_label
BENIGN    73.33
Fuzzy     26.67
fp_all_zero_rate: 0.9833
fp_id_zero_rate: 0.7333
tp_all_zero_rate: 0.887
tp_id_zero_rate: 0.8869
top_fp_ids
id
0       12222
1520     4165
688       278
291         1
top_fp_signatures
id    val0  val1  val2  val3  val4  val5  val6  val7  true_l

## Aggressive Filter: remove all BENIGN all-zero payloads
Rule:
- `flag == 'BENIGN'`
- `val0..val7` all zero

This is more aggressive than the previous filter (which also required `id == 0`).

In [17]:
aggressive_mask = (df['flag'] == 'BENIGN') & (df[byte_cols] == 0).all(axis=1)
df_aggressive = df.loc[~aggressive_mask].copy()

print('removed_benign_all_zero_rows:', int(aggressive_mask.sum()))
print('aggressive_shape:', df_aggressive.shape)
print('aggressive class counts')
print(df_aggressive['flag'].value_counts().to_string())

benign_df_a = df_aggressive[df_aggressive['flag'] == 'BENIGN']
attack_df_a = df_aggressive[df_aggressive['flag'] != 'BENIGN']
benign_sampled_a = benign_df_a.sample(n=min(BENIGN_TARGET, len(benign_df_a)), random_state=RANDOM_STATE)
df_balanced_a = pd.concat([benign_sampled_a, attack_df_a], axis=0)
df_balanced_a = df_balanced_a.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

df_balanced_a['flag_encoded'] = le.transform(df_balanced_a['flag'])
X_a = df_balanced_a[['id'] + byte_cols].copy()
y_a = df_balanced_a['flag_encoded'].copy()

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_a, y_a, test_size=0.2, stratify=y_a, random_state=40
)

mlp_aggressive_model, mlp_aggressive_pred, _ = train_mlp(
    X_train_a, X_test_a, y_train_a, y_test_a, 'MLP - id + bytes (remove all BENIGN all-zero)'
)

analysis_aggressive = build_error_frame(X_test_a, y_test_a.to_numpy(), mlp_aggressive_pred)
summarize_false_positives(analysis_aggressive, 'DoS', 'id + bytes + aggressive BENIGN all-zero removal')
summarize_false_positives(analysis_aggressive, 'GEAR', 'id + bytes + aggressive BENIGN all-zero removal')

removed_benign_all_zero_rows: 1581282
aggressive_shape: (17275465, 10)
aggressive class counts
flag
BENIGN    14668620
RPM         696373
DoS         693372
GEAR        637417
Fuzzy       579683
===== MLP - id + bytes (remove all BENIGN all-zero) =====
              precision    recall  f1-score   support

      BENIGN    0.99955   0.99950   0.99953    200000
         DoS    0.99991   0.95499   0.97693    138674
       Fuzzy    0.99923   0.97627   0.98762    115937
        GEAR    0.99997   0.93642   0.96716    127483
         RPM    0.89119   1.00000   0.94246    139275

    accuracy                        0.97616    721369
   macro avg    0.97797   0.97344   0.97474    721369
weighted avg    0.97872   0.97616   0.97653    721369

===== id + bytes + aggressive BENIGN all-zero removal | DoS =====
false_positives: 12
true_positives: 132432
actual_class_breakdown
true_label
BENIGN    12
actual_class_breakdown_pct
true_label
BENIGN    100.0
fp_all_zero_rate: 0.0
fp_id_zero_rate: 0.0
tp_al

## Aggressive Filter + Bytes Only (No ID)
Train again after removing all BENIGN all-zero rows, but now using only byte columns (without `id`).

In [18]:
# Reuse df_aggressive (BENIGN all-zero removed), now train with bytes only

df_balanced_ab = df_balanced_a.copy()

X_ab = df_balanced_ab[byte_cols].copy()
y_ab = df_balanced_ab['flag_encoded'].copy()

X_train_ab, X_test_ab, y_train_ab, y_test_ab = train_test_split(
    X_ab, y_ab, test_size=0.2, stratify=y_ab, random_state=40
)

mlp_aggressive_bytes_model, mlp_aggressive_bytes_pred, _ = train_mlp(
    X_train_ab, X_test_ab, y_train_ab, y_test_ab, 'MLP - bytes only (after aggressive BENIGN all-zero removal)'
)

# Use X_test_a as reference only to inspect id patterns in false positives
analysis_aggressive_bytes = build_error_frame(X_test_a, y_test_ab.to_numpy(), mlp_aggressive_bytes_pred)

summarize_false_positives(analysis_aggressive_bytes, 'DoS', 'bytes only + aggressive BENIGN all-zero removal')
summarize_false_positives(analysis_aggressive_bytes, 'GEAR', 'bytes only + aggressive BENIGN all-zero removal')

print('===== confusion summary: bytes only + aggressive BENIGN all-zero removal =====')
cm_ab = confusion_matrix(analysis_aggressive_bytes['true_label'], analysis_aggressive_bytes['pred_label'], labels=le.classes_)
cm_ab_df = pd.DataFrame(cm_ab, index=le.classes_, columns=le.classes_)
print(cm_ab_df.loc[['BENIGN', 'DoS', 'GEAR'], ['BENIGN', 'DoS', 'GEAR']])

===== MLP - bytes only (after aggressive BENIGN all-zero removal) =====
              precision    recall  f1-score   support

      BENIGN    0.99953   0.99996   0.99975    200000
         DoS    0.96951   0.95499   0.96219    138674
       Fuzzy    0.99994   0.94036   0.96923    115937
        GEAR    1.00000   0.93642   0.96717    127483
         RPM    0.89120   1.00000   0.94247    139275

    accuracy                        0.97052    721369
   macro avg    0.97204   0.96635   0.96816    721369
weighted avg    0.97299   0.97052   0.97081    721369

===== bytes only + aggressive BENIGN all-zero removal | DoS =====
false_positives: 4165
true_positives: 132432
actual_class_breakdown
true_label
Fuzzy    4165
actual_class_breakdown_pct
true_label
Fuzzy    100.0
fp_all_zero_rate: 1.0
fp_id_zero_rate: 0.0
tp_all_zero_rate: 0.887
tp_id_zero_rate: 0.8869
top_fp_ids
id
1520    4165
top_fp_signatures
id    val0  val1  val2  val3  val4  val5  val6  val7  true_label
1520  0     0     0     0 

## RPM Error Analysis
Inspect RPM false positives to understand why RPM precision is lower.

In [19]:
def summarize_rpm_errors(analysis_df, title):
    fp = analysis_df[(analysis_df['pred_label'] == 'RPM') & (analysis_df['true_label'] != 'RPM')].copy()
    tp = analysis_df[(analysis_df['pred_label'] == 'RPM') & (analysis_df['true_label'] == 'RPM')].copy()

    print(f'===== {title} | RPM =====')
    print('rpm_false_positives:', len(fp))
    print('rpm_true_positives:', len(tp))

    if fp.empty:
        print('Sem falsos positivos para RPM.')
        print()
        return

    print('fp_true_class_breakdown')
    print(fp['true_label'].value_counts().to_string())
    print('fp_true_class_breakdown_pct')
    print((fp['true_label'].value_counts(normalize=True) * 100).round(2).to_string())

    print('fp_all_zero_rate:', round(float((fp[byte_cols] == 0).all(axis=1).mean()), 4))
    print('fp_id_zero_rate:', round(float((fp['id'] == 0).mean()), 4))

    print('top_fp_ids')
    print(fp['id'].value_counts().head(10).to_string())

    print('top_fp_signatures')
    print(fp[['id'] + byte_cols + ['true_label']].value_counts().head(12).to_string())

    print('rpm_tp_all_zero_rate:', round(float((tp[byte_cols] == 0).all(axis=1).mean()), 4))
    print('rpm_tp_id_zero_rate:', round(float((tp['id'] == 0).mean()), 4))
    print()

summarize_rpm_errors(analysis_with_id, 'id + bytes')
summarize_rpm_errors(analysis_aggressive, 'id + bytes + remove BENIGN all-zero')
summarize_rpm_errors(analysis_aggressive_bytes, 'bytes only + remove BENIGN all-zero')

===== id + bytes | RPM =====
rpm_false_positives: 21266
rpm_true_positives: 139275
fp_true_class_breakdown
true_label
GEAR      8105
Fuzzy     6823
DoS       6242
BENIGN      96
fp_true_class_breakdown_pct
true_label
GEAR      38.11
Fuzzy     32.08
DoS       29.35
BENIGN     0.45
fp_all_zero_rate: 0.2004
fp_id_zero_rate: 0.0
top_fp_ids
id
1520    21168
1440       96
721         1
1657        1
top_fp_signatures
id    val0  val1  val2  val3  val4  val5  val6  val7  true_label
1520  1     0     0     0     0     0     0     0     GEAR          8105
                                                      DoS           6242
      0     0     0     0     0     0     0     0     Fuzzy         4165
      1     0     0     0     0     0     0     0     Fuzzy         2656
1440  0     0     0     0     0     0     0     0     BENIGN          96
721   96    54    15    219   58    15    21    243   Fuzzy            1
1657  1     48    23    251   103   28    12    122   Fuzzy            1
rpm_tp_al

## Remove Incomplete CARDT Rows (`dlc < 8`) and Retrain
This experiment removes incomplete frames from Car-Hacking source (`source == 'CARDT' and dlc < 8`), then retrains to evaluate the impact on DoS/GEAR/RPM.

In [20]:
# Reload full aligned dataset with dlc/source to remove incomplete CARDT rows
full_df = pd.read_csv(DATA_PATH)
full_df = full_df[['id', 'dlc', 'val0', 'val1', 'val2', 'val3', 'val4', 'val5', 'val6', 'val7', 'flag', 'source']].dropna().copy()

incomplete_mask = (full_df['source'] == 'CARDT') & (full_df['dlc'] < 8)
print('incomplete CARDT rows to remove:', int(incomplete_mask.sum()))

df_no_incomplete = full_df.loc[~incomplete_mask].copy()
print('shape after remove incomplete:', df_no_incomplete.shape)
print('class counts after remove incomplete')
print(df_no_incomplete['flag'].value_counts().to_string())

# Balance BENIGN as before
benign_ni = df_no_incomplete[df_no_incomplete['flag'] == 'BENIGN']
attack_ni = df_no_incomplete[df_no_incomplete['flag'] != 'BENIGN']
benign_sampled_ni = benign_ni.sample(n=min(BENIGN_TARGET, len(benign_ni)), random_state=RANDOM_STATE)
df_balanced_ni = pd.concat([benign_sampled_ni, attack_ni], axis=0)
df_balanced_ni = df_balanced_ni.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

le = LabelEncoder()
df_balanced_ni['flag_encoded'] = le.fit_transform(df_balanced_ni['flag'])

X_ni = df_balanced_ni[['id'] + byte_cols].copy()
y_ni = df_balanced_ni['flag_encoded'].copy()

X_train_ni, X_test_ni, y_train_ni, y_test_ni = train_test_split(
    X_ni, y_ni, test_size=0.2, stratify=y_ni, random_state=40
)

mlp_no_incomplete_model, mlp_no_incomplete_pred, _ = train_mlp(
    X_train_ni, X_test_ni, y_train_ni, y_test_ni, 'MLP - id + bytes (remove CARDT incomplete dlc<8)'
)

analysis_no_incomplete = build_error_frame(X_test_ni, y_test_ni.to_numpy(), mlp_no_incomplete_pred)

summarize_false_positives(analysis_no_incomplete, 'DoS', 'id + bytes + remove CARDT incomplete')
summarize_false_positives(analysis_no_incomplete, 'GEAR', 'id + bytes + remove CARDT incomplete')
summarize_rpm_errors(analysis_no_incomplete, 'id + bytes + remove CARDT incomplete')

print('===== confusion summary: id + bytes + remove CARDT incomplete =====')
cm_ni = confusion_matrix(analysis_no_incomplete['true_label'], analysis_no_incomplete['pred_label'], labels=le.classes_)
cm_ni_df = pd.DataFrame(cm_ni, index=le.classes_, columns=le.classes_)
print(cm_ni_df.loc[['BENIGN', 'DoS', 'GEAR', 'RPM'], ['BENIGN', 'DoS', 'GEAR', 'RPM']])

incomplete CARDT rows to remove: 1189537
shape after remove incomplete: (17667210, 12)
class counts after remove incomplete
flag
BENIGN    15261030
DoS         662184
RPM         654897
GEAR        597252
Fuzzy       491847
===== MLP - id + bytes (remove CARDT incomplete dlc<8) =====
              precision    recall  f1-score   support

      BENIGN    0.99962   0.99977   0.99969    200000
         DoS    1.00000   0.99998   0.99999    132437
       Fuzzy    0.99952   0.99924   0.99938     98370
        GEAR    1.00000   1.00000   1.00000    119450
         RPM    0.99999   1.00000   1.00000    130979

    accuracy                        0.99982    681236
   macro avg    0.99983   0.99980   0.99981    681236
weighted avg    0.99982   0.99982   0.99982    681236

===== id + bytes + remove CARDT incomplete | DoS =====
false_positives: 0
true_positives: 132434
Sem falsos positivos para essa classe.

===== id + bytes + remove CARDT incomplete | GEAR =====
false_positives: 0
true_positives